### Задание 2 - 30 баллов

1. Построить и оценить качество бейзлайна

В рамках данного пункта необходимо выбрать и обучить бейзлайн-модели, а также измерить их качество.

Критерии оценки:
- Произведено разбиение датасета на тренировочную/тестовую выборки - **2 балла**
- Произведено измерение качества константного предсказания (например, наиболее частотный класс для классификации, среднее/медиана для регрессии) - **3 балла**
- Бейзлайновая модель из простого семейства (линейные модели, деревья решений, knn...) обучена на тренировочной выборке, учтены особенности предобработки данных для модели, если они есть - **6 баллов**
- Произведено измерение качества на отложенной выборке с использованием ранее выбранной метрики - **2 балла**
- Обеспечена воспроизводимость решения: зафиксированы `random_state`, ноутбук воспроизводится от начала до конца без ошибок - **3 балла**
- Соблюден code style на уровне [pep8](https://peps.python.org/pep-0008/) и [On writing clean Jupyter notebooks](https://ploomber.io/blog/clean-nbs/) - **4 балла**
- Принимаемые решения обоснованы и прокомментированы в markdown ячейках (то есть, например, если для кодирования категориальных переменных выбран метод Label Encoding, то текстом написано, почему он, и тп) - **10 баллов**

Ссылка на данные: https://www.kaggle.com/datasets/rishikeshkonapure/hr-analytics-prediction/data

In [1]:
# Импортируем библиотеки
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,MinMaxScaler,StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score ,classification_report

Теперь нам необходимо загрузить наши данные и выполнить очистку для удобной работы с датасетом, как было сделано в предыдущей работе

In [2]:
df=pd.read_csv('HR-Employee-Attrition.csv')
df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,...,3,80,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7


Мы можем увидеть что ряд значений так EmployeeCount, EmployeeNumber, Over18,StandardHours имеют значения которые не являются уникальными или имеют всего 2 значения, которые не несут в себе существенной для дальнейшего анализа информации, поэтому мы удалим их

In [3]:
df=df.drop(['EmployeeCount','EmployeeNumber','Over18','StandardHours'], axis=1)
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,3,Male,...,3,3,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,4,Male,...,3,1,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,2,Male,...,4,2,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,4,Male,...,3,4,0,17,3,2,9,6,0,8


1) Преобразование датасета
---



Необходимо произвести преобразование датасета так как не все значения воспользуемся функцией LabelEncoder

Для целевой переменной 'Attrition' значение сотвественно 1 = Yes  0 = No

In [4]:
# Выполянем преобразование объектов 'Object' в числовой формат 'Int'
le = LabelEncoder()

for i in df.columns:
    if df[i].dtype == 'object':
        df[i] = le.fit_transform(df[i])
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,2,1102,2,1,2,1,2,0,...,3,1,0,8,0,1,6,4,0,5
1,49,0,1,279,1,8,1,1,3,1,...,4,4,1,10,3,3,10,7,1,7
2,37,1,2,1373,1,2,2,4,4,1,...,3,2,0,7,3,3,0,0,0,0
3,33,0,1,1392,1,3,4,1,4,0,...,3,3,0,8,3,3,8,7,3,0
4,27,0,2,591,1,2,1,3,1,1,...,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,0,1,884,1,23,2,3,3,1,...,3,3,1,17,3,3,5,2,0,3
1466,39,0,2,613,1,6,1,3,4,1,...,3,1,1,9,5,3,7,7,1,7
1467,27,0,2,155,1,4,3,1,2,1,...,4,2,1,6,0,3,6,2,0,3
1468,49,0,1,1023,2,2,3,3,4,1,...,3,4,0,17,3,2,9,6,0,8


2) Разбиение выборки на тренировочную/тестовую
---
Перед обучением модели разделим тренировочную и тестовую выборки. Воспользуемся функцией `train_test_split`
Для обеспечения воспроизводимости модели установим параметр `random_state` = 8


In [5]:
# Разобьем датасет исключив целевую переменную из одного из них
x = df.drop(columns=['Attrition'])
y = df['Attrition']

#Разделим выборку на train и test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=8)

print(f'train: {len(x_train)}, test: {len(x_test)}')

train: 1176, test: 294


3) Обучение моделей
---

После того как мы разделили выборку, начнем обучение модели с `DummyClassifier` опеределим минимальный уровень качества модели и получим контрольные данные для сравнительной оценки более сложной модели построенной в будущем

In [6]:
dumclass = DummyClassifier(strategy="most_frequent")
dumclass.fit(x_train, y_train)

y_pred = dumclass.predict(x_test)
print(classification_report(y_test, y_pred))

#Для оценки модели в качестве выбранной метрики используем Accuracy
dummy_accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {dummy_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.85      1.00      0.92       251
           1       0.00      0.00      0.00        43

    accuracy                           0.85       294
   macro avg       0.43      0.50      0.46       294
weighted avg       0.73      0.85      0.79       294

Accuracy: 0.8537


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Попробуем использовать Дерево решений для обучения нашей модели

In [7]:
DecTree = DecisionTreeClassifier()
DecTree.fit(x_train, y_train)

y_preds_dt = DecTree.predict(x_test)
print(classification_report(y_test, y_preds_dt))

#Для оценки модели в качестве выбранной метрики используем Accuracy
DecTree_accuracy = accuracy_score(y_test, y_preds_dt)
print(f'Accuracy: {DecTree_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.89      0.85      0.87       251
           1       0.31      0.40      0.35        43

    accuracy                           0.78       294
   macro avg       0.60      0.62      0.61       294
weighted avg       0.81      0.78      0.79       294

Accuracy: 0.7823


Результаты получились не самыми лучшими Accuracy: 0.7721 ниже чем DummyClassifier, это может быт связано с отсутствим глубины дерева и иных параметров, попробуем добавить их

In [8]:
DecTree = DecisionTreeClassifier(criterion='entropy', max_depth=3)
DecTree.fit(x_train, y_train)

y_preds_dt = DecTree.predict(x_test)
print(classification_report(y_test, y_preds_dt))

#Для оценки модели в качестве выбранной метрики используем Accuracy
DecTree_accuracy = accuracy_score(y_test, y_preds_dt)
print(f'Accuracy: {DecTree_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.87      1.00      0.93       251
           1       0.83      0.12      0.20        43

    accuracy                           0.87       294
   macro avg       0.85      0.56      0.57       294
weighted avg       0.86      0.87      0.82       294

Accuracy: 0.8673


Теперь попробуем построить логистическую регрессию и посмотрим на результат

In [9]:
LorReg = LogisticRegression(random_state=8)
LorReg.fit(x_train, y_train)

y_pred_lr = LorReg.predict(x_test)
print(classification_report(y_test, y_pred_lr))

#Для оценки модели в качестве выбранной метрики используем Accuracy
LorReg_accuracy = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy: {LorReg_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.85      1.00      0.92       251
           1       0.00      0.00      0.00        43

    accuracy                           0.85       294
   macro avg       0.43      0.50      0.46       294
weighted avg       0.73      0.85      0.79       294

Accuracy: 0.8537


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Упс, похоже есть некоторая проблема в определении целевой метрики в модели, попробуем провести стандартизацию и посмотреть как изменятся результаты, возможно данные были плохо подготовлены

In [11]:
# Проведем стандартизацию данных
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

LorReg = LogisticRegression(random_state=8)
LorReg.fit(x_train_scaled, y_train)
print(classification_report(y_test, y_pred_lr))

y_pred_lr = LorReg.predict(x_test_scaled)
LorReg_accuracy = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy после стандартизации: {LorReg_accuracy:.4f}')


              precision    recall  f1-score   support

           0       0.89      0.96      0.92       251
           1       0.56      0.33      0.41        43

    accuracy                           0.86       294
   macro avg       0.73      0.64      0.67       294
weighted avg       0.84      0.86      0.85       294

Accuracy после стандартизации: 0.8639


Результаты намного лучше, теперь модель повысила показатели метрики

Выводы:
---

Мы обучили несоклько моделей и получили неплохие результатыЖ, методом проб и ошибок удалось создать модель, возможно чуть переобученное деревье решений

Метрики у логистической регрессии получше, но нужно будет подумать как сделать их лучше в будущем
